# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ujjwalupreti/flyrank-internship-capstone/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This playbook provides decision-support by identifying pages that look worth reviewing first. We use the validated model to output a ranked queue, rather than automating a final decision. Each page flagged by the model is assigned a reason code based on its archetype, providing concrete context for the human reviewer. The model flags these actions at a specific precision@K, allowing us to focus only on the highest-confidence recommendations.

In [1]:
import pandas as pd
import numpy as np

# Simulate loading the validated dataset and model predictions from Week 5
# In practice, this would be: df_queue = pd.read_csv('work/outputs/scored_test_set.csv')
data = {
    'page_path': ['/blog/seo-tips', '/pricing', '/about', '/blog/old-news', '/features'],
    'max_impressions': [8500, 12000, 400, 50, 6000],
    'estimated_staleness_days': [180, 15, 300, 400, 120],
    'model_score': [0.88, 0.05, 0.65, 0.91, 0.78]
}
df_queue = pd.DataFrame(data)

# Define archetype -> action mapping (Reason Codes)
def assign_reason_code(row):
    if row['model_score'] >= 0.75 and row['max_impressions'] >= 5000:
        return "ACTION: High-Priority Refresh (Decaying High-Volume)"
    elif row['model_score'] >= 0.75 and row['max_impressions'] < 5000:
        return "REVIEW: Stale but Low-Volume (Check ROI before refresh)"
    elif row['model_score'] >= 0.50:
        return "MONITOR: Approaching staleness threshold"
    else:
        return "NO-GO: Stable content, do not touch"

df_queue['reason_code'] = df_queue.apply(assign_reason_code, axis=1)

# Sort by model score to create the ranked queue
df_queue = df_queue.sort_values(by='model_score', ascending=False).reset_index(drop=True)
display(df_queue)

,page_path,max_impressions,estimated_staleness_days,model_score,reason_code
0,/blog/old-news,50,400,0.91,REVIEW: Stale but Low-Volume (Check ROI before...
1,/blog/seo-tips,8500,180,0.88,ACTION: High-Priority Refresh (Decaying High-V...
2,/features,6000,120,0.78,ACTION: High-Priority Refresh (Decaying High-V...
3,/about,400,300,0.65,MONITOR: Approaching staleness threshold
4,/pricing,12000,15,0.05,"NO-GO: Stable content, do not touch"


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

* **Intended Use:** This playbook is designed to support the decision-making of content strategists.


* **Limitations & Honest Framing:** We observed these specific decay patterns within this dataset and period alone.


* Because this analysis relies on cross-sectional data without a controlled intervention, it never supports claims that executing a refresh *causes* or *will increase* traffic.


* We must also account for survivorship bias; if inactive pages were pre-filtered out of our data, we must state this filter when presenting the findings.




In [2]:
# Calculate basic limit metrics to guide the user
total_pages = len(df_queue)
actionable_pages = len(df_queue[df_queue['model_score'] >= 0.75])
base_rate_proxy = actionable_pages / total_pages

print("--- Playbook Limits & Scope ---")
print(f"Total Pages Analyzed: {total_pages}")
print(f"Actionable Queue (Score >= 0.75): {actionable_pages} pages")
print(f"Actionable Base Rate: {base_rate_proxy:.2f}")
print("Note: Applying this model to pages outside the original training distribution (e.g., brand new domains) voids these limits.")

--- Playbook Limits & Scope ---
Total Pages Analyzed: 5
Actionable Queue (Score >= 0.75): 3 pages
Actionable Base Rate: 0.60
Note: Applying this model to pages outside the original training distribution (e.g., brand new domains) voids these limits.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### 3. Human review + the no-go list

**Paste this into the Markdown cell:**

* **Human Review Required:** All actions require human verification, as our analysis is an observation of historical patterns, not a controlled experiment.


* Editors must check for selection bias in the recommendations; if the "treated" pages in our historical data were manually cherry-picked by a team in the past, part of the historical performance gap is due to that choosing, not just the refresh itself.


* **The No-Go List:** Any automated, programmatic execution based on these scores is strictly banned.


* We exclude pages with zero business value (e.g., privacy policies) from the action queue to maintain practical cost/value thinking.


In [3]:
# Define the strict No-Go filters based on human-review rules
def flag_nogo_cases(row):
    if "NO-GO" in row['reason_code']:
        return True
    # Prevent automation on legal/admin pages regardless of score
    if any(term in row['page_path'] for term in ['privacy', 'terms', 'about']):
        return True
    return False

df_queue['is_nogo'] = df_queue.apply(flag_nogo_cases, axis=1)

# Generate the final human-filtered queue
human_review_queue = df_queue[df_queue['is_nogo'] == False].copy()
print("--- Final Queue for Human Review (No-Go cases removed) ---")
display(human_review_queue[['page_path', 'model_score', 'reason_code']])

--- Final Queue for Human Review (No-Go cases removed) ---


,page_path,model_score,reason_code
0,/blog/old-news,0.91,REVIEW: Stale but Low-Volume (Check ROI before...
1,/blog/seo-tips,0.88,ACTION: High-Priority Refresh (Decaying High-V...
2,/features,0.78,ACTION: High-Priority Refresh (Decaying High-V...


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

* To ensure the model remains honest, we must monitor the baseline accuracy alongside its base rate.


* If the base rate of decaying content shifts dramatically (e.g., due to a Google core update), the model must be retrained to reflect the new reality.


* When reviewing the model's ongoing performance, we frame findings as "how to make it stronger" rather than treating failures as gotchas.






In [4]:
# Define a lightweight monitoring check
TRAINING_BASE_RATE = 0.35 # Simulated base rate from Week 5

def check_drift_trigger(current_base_rate, threshold=0.10):
    drift = abs(current_base_rate - TRAINING_BASE_RATE)
    if drift > threshold:
        return f"WARNING: Retrain Triggered. Base rate drifted by {drift:.2f} (Threshold: {threshold})"
    return "STATUS OK: Base rate within acceptable bounds."

print(check_drift_trigger(base_rate_proxy))

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

* The exported ranked recommendations represent the practical "so what" of this analysis.


* We export the queue CSV to `work/outputs/` to ensure the data is public-safe and kept out of git by design.


* We also export canonical charts to `work/figures/` to guarantee reproducibility and allow a stranger to trust the workflow.



In [5]:
import os
import matplotlib.pyplot as plt

# Ensure directories exist
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# Export the Ranked Queue (stays out of git)
export_path = 'work/outputs/playbook_action_queue.csv'
human_review_queue.to_csv(export_path, index=False)
print(f"Queue exported to: {export_path}")

# Generate and export a figure for the research paper
plt.figure(figsize=(8, 4))
plt.barh(human_review_queue['page_path'], human_review_queue['model_score'], color='teal')
plt.xlabel('Model Confidence Score')
plt.title('Top Pages Flagged for Content Refresh')
plt.gca().invert_yaxis() # Highest score at top

fig_path = 'work/figures/ranked_recommendations.png'
plt.savefig(fig_path, bbox_inches='tight')
plt.close()
print(f"Chart exported to: {fig_path}")

Queue exported to: work/outputs/playbook_action_queue.csv
Chart exported to: work/figures/ranked_recommendations.png


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.